# 🎯 [OPS02_M07] 중복 항적 강제 분리 및 통합 확정을 위한 조작관 HITL 검토/승인 체계 구축
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS02/ops02_m07_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS02_M07 |
| **미션 주제** | 중복 항적 강제 분리 및 통합 확정을 위한 조작관 HITL 검토/승인 체계 구축 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
미국-이란 간 무력 충돌 사례에서는 드론·미사일이 여러 방향에서 동시에 들어오면서 레이더, 광학(EO/IR), 음향 센서가 같은 비행체를 각자 경보해 상황판에 중복 항적이 쏟아지고, 이 때문에 위협 수를 잘못 세거나 대응이 늦어지는 문제가 주목받았다. 본 프로젝트에서는 가상 방어 지역에 배치한 가상 센서 4기와 가상 비행체 궤적으로 교육용 경보 데이터를 코드로 만들어, 연동 규격과 통합 스키마 설계, 시각 동기화와 ENU 좌표 변환 전처리, 최단거리 매칭 Baseline, 확장 칼만 필터(EKF) 항적 추적·통합, 융합 신뢰도 지수, 전술 상황판(COP), 조작관 HITL 검토, 통신 단절·재밍 스트레스 시험, AAR 재보정, 최종 통합까지 다중 센서 중복 항적 통합 시뮬레이터를 단계별로 구현한다. 최종 목표는 비행체 하나를 하나의 항적으로 정확히 보여 주어 지휘관의 인지 부하를 줄이고, 항적 통합·분리 같은 판단은 조작관이 근거와 함께 확정하도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/ops02_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/ui/`, `src/hitl/` | 전처리, 모델, 대시보드, HITL 승인 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/ops02_data_spec.md`, `docs/ops02_hitl_guide.md`, `docs/ops02_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서 등 실무/지휘관 문서 |
| **`configs/`**| `configs/ops02_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/ops02_stress_test.py` | 500% 부하 시험 및 pytest 단위/회귀 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (추천):** 코드 셀 첫 줄에 `%%writefile src/models/ops02_baseline_model.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 07. 중복 항적 강제 분리 및 통합 확정을 위한 조작관 HITL 검토/승인 체계 구축
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

센서 간 상충 정보 발생 시 조작관이 직접 항적 분리, 강제 통합, 추적 보류를 결정할 수 있는 HITL 검토/승인 체계를 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **센서 간 상충 정보로 생긴 항적 충돌을 조작관이 직접 통합·분리·보류·유지로 결정하는 HITL 검토/승인 체계를 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **융합 엔진은 중복 의심, 병합 의심, 단일 센서 항적을 충돌로 제안만 하고 항적 상태는 바꾸지 않는다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/hitl/ops02_approval_workflow.py 및 docs/ops02_hitl_guide.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 지휘 책임 분계선 명확화 및 성급한 자동 단정 방지 체계 부합성

#### 미션 PASS 조건

- 정량 기준: 인간 검토 큐(Queue) 대기시간 및 승인/반려 처리 로깅 성공률 100%


- 진행 기준: 승인/반려 인터페이스 동작 및 감사 로그(Audit Log) 생성 검증


- 검증 방법: 모의 조작자 승인/반려 시뮬레이션 및 데이터 무결성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
# [TODO : 학생 실습 7] 중복 항적 강제 분리 및 통합 확정을 위한 조작관 HITL 검토/승인 체계 구축
# 미션 지침 및 검증 기준을 확인하고 코드를 실행 및 튜닝하세요.
"""센서 간 상충 정보로 생긴 항적 충돌(중복 의심, 병합 의심, 단일 센서 항적)을 조작관 검토 대기열에 올리고, 강제 통합·분리·추적 보류·현행 유지를 조작관이 근거와 함께 직접 결정하도록 하는 HITL 검토/승인 체계와 해시 체인 감사 로그를 구현한다."""
import hashlib
import json
import numpy as np
import pandas as pd

ACTIONS = {"MERGE": "강제 통합", "SPLIT": "항적 분리", "HOLD": "추적 보류(감시 유지)", "KEEP": "현행 유지"}
SEVERITY = {"DUP": 0, "MIXED": 1, "SINGLE": 2}                    # 중복 의심 > 병합 의심 > 단일 센서
HOLD_LIMIT_S = 60
CENTER = np.array([8.0, 2.0])                                       # 가상 방어 지역 중심(km)


class WorkflowError(ValueError):
    pass


class ReviewQueue:
    def __init__(self, tracks, clock):
        self.tracks, self.items, self.log, self.clock, self.feedback = tracks, {}, [], clock, []

    def _record(self, body):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {**body, "ts": round(self.clock(), 1)}
        h = hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest()
        self.log.append({**body, "prev": prev, "hash": h})

    def verify_log(self):
        prev = "GENESIS"
        for n, rec in enumerate(self.log, start=1):
            body = {k: v for k, v in rec.items() if k not in ("prev", "hash")}
            if rec["prev"] != prev or rec["hash"] != hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest():
                return False, n
            prev = rec["hash"]
        return True, -1

    def submit(self, cid, kind, tids, note):
        """시스템은 충돌을 '제안'만 하고 항적 상태는 바꾸지 않는다(PENDING)."""
        d = min(np.hypot(*(np.array(self.tracks[t]["pos"]) - CENTER)) for t in tids)
        self.items[cid] = {"id": cid, "kind": kind, "tids": tids, "note": note, "dist_km": round(float(d), 1), "status": "PENDING",
                           "submitted": self.clock(), "hold_since": None, "escalated": False}
        self._record({"event": "SUBMIT", "id": cid, "kind": kind, "tids": tids, "by": "FUSION"})

    def pending(self):
        todo = [i for i in self.items.values() if i["status"] in ("PENDING", "HOLD")]
        return sorted(todo, key=lambda i: (not i["escalated"], SEVERITY[i["kind"]], i["dist_km"], i["submitted"]))

    def decide(self, cid, action, operator, reason):
        if action not in ACTIONS:
            raise WorkflowError(f"허용되지 않은 조치: {action}")
        if not operator:
            raise WorkflowError("조작관 ID 필요")
        it = self.items.get(cid)
        if it is None or it["status"] in ("DONE",):
            raise WorkflowError("대기열에 없거나 이미 종결된 충돌")
        if len(reason.strip()) < 5:
            raise WorkflowError("판단 근거 5자 이상 필요")
        T = [self.tracks[t] for t in it["tids"]]
        if action == "MERGE":
            if len(T) < 2:
                raise WorkflowError("통합은 항적 2개 이상 필요")
            gap = np.hypot(*(np.array(T[0]["pos"]) - np.array(T[1]["pos"])))
            if gap > 1.0 and len(reason.strip()) < 15:
                raise WorkflowError(f"간격 {gap:.1f}km 항적 통합은 근거 15자 이상 필요(오통합 방지)")
            T[1]["status"] = f"MERGED->{T[0]['id']}"; T[0]["sensors"] = sorted(set(T[0]["sensors"]) | set(T[1]["sensors"]))
        elif action == "SPLIT":
            if len(T[0]["sensors"]) < 2:
                raise WorkflowError("단일 센서 항적은 분리 대상이 아님")
            new = f"{T[0]['id']}b"
            self.tracks[new] = {"id": new, "pos": T[0]["pos"], "sensors": T[0]["sensors"][1:], "status": "SPLIT"}
            T[0]["sensors"] = T[0]["sensors"][:1]
        elif action == "HOLD":
            for t in T:
                t["status"] = "HOLD"
        it["status"] = "HOLD" if action == "HOLD" else "DONE"
        it["hold_since"] = self.clock() if action == "HOLD" else None
        it["decided"], it["action"] = self.clock(), action
        self._record({"event": "DECIDE", "id": cid, "action": action, "operator": operator, "reason": reason})

    def check_holds(self):
        out = []
        for i in self.items.values():
            if i["status"] == "HOLD" and not i["escalated"] and self.clock() - i["hold_since"] > HOLD_LIMIT_S:
                i["escalated"] = True; out.append(i["id"])
                self._record({"event": "ESCALATE", "id": i["id"], "to": "방공 상황장교", "by": "SYSTEM"})
        return out

    def commander_feedback(self, cid, text):
        self.feedback.append({"id": cid, "text": text}); self._record({"event": "FEEDBACK", "id": cid, "text": text})


class FakeClock:
    t = 0.0

    def __call__(self):
        return self.t


# [교육용가상자료] 융합 엔진이 넘긴 가상 항적과 충돌 6건
tracks = {t: {"id": t, "pos": p, "sensors": s, "status": "ACTIVE"} for t, p, s in [
    ("FT003", (9.1, 3.0), ["RDR-A", "RDR-B"]), ("FT007", (9.4, 3.2), ["EOIR-C", "ACU-D"]), ("FT010", (14.0, -3.0), ["RDR-A", "RDR-B", "EOIR-C"]),
    ("FT012", (3.5, 12.0), ["EOIR-C"]), ("FT015", (12.0, 6.0), ["RDR-A"]), ("FT016", (13.5, 6.5), ["RDR-B"]), ("FT020", (6.0, 0.5), ["RDR-A", "ACU-D"])]}
clock = FakeClock()
q = ReviewQueue(tracks, clock)
for cid, kind, tids, note in [("C01", "DUP", ["FT003", "FT007"], "0.4km 간격, 속도 유사"), ("C02", "MIXED", ["FT010"], "EO/IR 방위가 레이더와 2도 어긋남"),
                              ("C03", "SINGLE", ["FT012"], "EO/IR 단독 30초 유지"), ("C04", "DUP", ["FT015", "FT016"], "1.6km 간격, 레이더별 단독"),
                              ("C05", "SINGLE", ["FT020"], "음향 소실 후 레이더 단독"), ("C06", "MIXED", ["FT020"], "음향·레이더 위치 불일치")]:
    clock.t += 3; q.submit(cid, kind, tids, note)
print("검토 순서(충돌 유형·방어 지역 거리 순):", [i["id"] for i in q.pending()])
print("자동 적용된 충돌 수:", sum(i["status"] != "PENDING" for i in q.items.values()))

blocked = 0
for cid, a, op, r in [("C01", "MERGE", "", "근거 충분"), ("C02", "AUTO", "op1", "자동 처리"), ("C03", "SPLIT", "op1", "단독 항적 분리 시도"),
                      ("C04", "MERGE", "op1", "위치 유사"), ("C05", "KEEP", "op1", "확인")]:
    try:
        q.decide(cid, a, op, r)
    except WorkflowError as e:
        blocked += 1; print(f"차단({cid} {a}): {e}")

clock.t += 10; q.decide("C01", "MERGE", "op1", "속도·고도 추정 일치, 같은 비행체로 판단")
clock.t += 6; q.decide("C02", "SPLIT", "op2", "EO/IR 영상상 2기 근접 비행 확인")
clock.t += 5; q.decide("C04", "MERGE", "op1", "레이더 A·B 시계 보정 후 궤적 일치, 동일 표적으로 판단")
clock.t += 4; q.decide("C03", "HOLD", "op2", "기만 가능성, 레이더 재확인 요청")
clock.t += 3; q.decide("C05", "KEEP", "op2", "레이더 연속 추적 정상")
clock.t += 90
print("보류 기한 초과 상신:", q.check_holds(), "/ 상신 후 검토 순서:", [i["id"] for i in q.pending()])
q.commander_feedback("C03", "EO/IR 단독 항적은 30초 이상 지속 시 상황장교에게 즉시 보고")

df = pd.DataFrame(q.items.values())
df["대기(s)"] = (df["decided"] - df["submitted"]).round(1)
print("\n" + df[["id", "kind", "tids", "dist_km", "status", "action", "escalated", "대기(s)"]].fillna("-").to_string(index=False))
print("\n항적 상태:", {k: v["status"] for k, v in tracks.items()})
n_dec = sum(r["event"] == "DECIDE" for r in q.log)
n_auto = sum(r["event"] == "DECIDE" and not r.get("operator") for r in q.log)
n_exp = sum("decided" in i for i in q.items.values())
print(f"\n자동 적용 {n_auto}건 / 규칙 위반 차단 {blocked}건 / 결정 로그 {n_dec}건(결정 {n_exp}건, 로깅 성공률 {n_dec / n_exp:.0%})")
print(f"평균 검토 대기 {df['대기(s)'].mean():.1f}s, 감사 로그 {len(q.log)}건, 해시 체인 검증: {q.verify_log()}")
q.log[6]["action"] = "KEEP"
print("7번째 기록 변조 후 검증(False, 깨진 위치):", q.verify_log())
print("\n책임 분계: 융합 엔진=충돌 제안, 조작관=통합·분리·보류 결정, 상황장교=상신 건 판단. 항적 결정은 교전 결정이 아니며 교전은 교전수칙·지휘계통에 따른다.")


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
검토 순서(충돌 유형·방어 지역 거리 순): ['C01', 'C04', 'C06', 'C02', 'C05', 'C03']
자동 적용된 충돌 수: 0
차단(C01 MERGE): 조작관 ID 필요
차단(C02 AUTO): 허용되지 않은 조치: AUTO
차단(C03 SPLIT): 단일 센서 항적은 분리 대상이 아님
차단(C04 MERGE): 간격 1.6km 항적 통합은 근거 15자 이상 필요(오통합 방지)
차단(C05 KEEP): 판단 근거 5자 이상 필요
보류 기한 초과 상신: ['C03'] / 상신 후 검토 순서: ['C03', 'C06']

 id   kind           tids  dist_km  status action  escalated 대기(s)
C01    DUP [FT003, FT007]      1.5    DONE  MERGE      False  25.0
C02  MIXED        [FT010]      7.8    DONE  SPLIT      False  28.0
C03 SINGLE        [FT012]     11.0    HOLD   HOLD       True  34.0
C04    DUP [FT015, FT016]      5.7    DONE  MERGE      False  27.0
C05 SINGLE        [FT020]      2.5    DONE   KEEP      False  31.0
C06  MIXED        [FT020]      2.5 PENDING      -      False     -

항적 상태: {'FT003': 'ACTIVE', 'FT007': 'MERGED->FT003', 'FT010': 'ACTIVE', 'FT012': 'HOLD', 'FT015': 'ACTIVE', 'FT016': 'MERGED->FT015', 'FT020': 'ACTIVE', 'FT010b': 'SPLIT'}

자동 적용 0건 / 규칙 위반 차단 5건 / 결정 로그 5건(결정 5건, 로깅 성공률 100%)
평균 검토 대기 29.0s, 감사 로그 13건, 해시 체인 검증: (True, -1)
7번째 기록 변조 후 검증(False, 깨진 위치): (False, 7)

책임 분계: 융합 엔진=충돌 제안, 조작관=통합·분리·보류 결정, 상황장교=상신 건 판단. 항적 결정은 교전 결정이 아니며 교전은 교전수칙·지휘계통에 따른다.
```
</details>
